# 🏔️ Semana 12 · Unidad 3 — Heapsort

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 12 · Unidad 3 — Heapsort |
| **Duración** | 50 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import random
import time
from IPython.display import display, HTML
print("✅ Dependencias cargadas correctamente")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Describir** las dos fases de Heapsort: heapify y sortdown.
2. **Implementar** Heapsort in-place usando un arreglo 1-based.
3. **Explicar** por qué Heapsort construye el heap usando sink (no swim) en la fase 1.
4. **Analizar** la complejidad de Heapsort: O(n log n) en el peor caso con O(1) de espacio extra.
5. **Comparar** Heapsort con Mergesort y Quicksort, identificando cuándo usar cada uno.

# Sección 1: Del Heap al Ordenamiento (8 minutos)

## La observación de la clase anterior

Al final de la Clase 6 vimos que si extraemos todos los elementos de un max-heap con `delMax()`, salen **en orden decreciente**.

```python
pq = MaxHeap()
for v in [5, 3, 8, 1, 7, 2, 9, 4, 6]:
    pq.insert(v)

while not pq.isEmpty():
    print(pq.delMax())   # imprime: 9, 8, 7, 6, 5, 4, 3, 2, 1
```

Eso es una ordenación. ¿Podemos hacer esto in-place?

## El problema del enfoque directo

Si usamos la MaxHeap de la clase anterior:
- Necesitamos O(n) de memoria extra para el heap.
- Tenemos que copiar los elementos al heap y luego al arreglo resultado.

**Heapsort resuelve esto** usando el arreglo original como el heap.

## Las dos fases de Heapsort

```
Arreglo desordenado
        ↓
  Fase 1: heapify  →  construir max-heap in-place (el arreglo es el heap)
        ↓
  Fase 2: sortdown →  extraer máximos sucesivos al final del arreglo
        ↓
Arreglo ordenado (ascendente)
```

> 🎙️ **[PAUSA PROFESOR]** *"Si extraemos el máximo y lo ponemos al final, y hacemos eso n veces... ¿dónde va quedando el arreglo ordenado?"*  
> *Respuesta: el arreglo ordenado crece desde el final. Al final, todo el arreglo está ordenado ascendentemente.*

# Sección 2: Fase 1 — Heapify (15 minutos)

## ¿Cómo construir el heap in-place?

**Opción A (ingenua):** Insertar elemento a elemento con swim.  
Costo: n × O(log n) = O(n log n)

**Opción B (eficiente, la de Heapsort):** Aplicar sink de derecha a izquierda, **saltando las hojas**.  
Costo: O(n) — ¡lineal!

## La idea de heapify con sink

Las **hojas** ya satisfacen la propiedad heap (no tienen hijos, no pueden violarla).  
El último nodo interno tiene índice `n // 2`.

```
Arreglo inicial: [_, 4, 1, 3, 2, 9, 7, 8, 5, 6]  (índice 0 sin usar)
                      ↑                ↑
               n//2 = 4               n = 9

Árbol:
         4(1)
        /    \
      1(2)   3(3)
     /  \   /  \
   2(4) 9(5)7(6) 8(7)
   / \
 5(8) 6(9)

Aplicar sink desde posición n//2=4 hasta 1:

sink(4): heap[4]=2, hijos: heap[8]=5, heap[9]=6  → 2 < 6, swap(4,9) → ...
sink(3): heap[3]=3, hijos: heap[6]=7, heap[7]=8  → 3 < 8, swap(3,7) → ...
sink(2): heap[2]=1, hijos: heap[4]=6, heap[5]=9  → 1 < 9, swap(2,5) → ...
sink(1): heap[1]=4, hijos: heap[2]=9, heap[3]=8  → 4 < 9, swap(1,2) → ...
```

## ¿Por qué heapify con sink es O(n) y no O(n log n)?

Las hojas (aprox. n/2 nodos) no hacen ningún intercambio.  
Los nodos en el penúltimo nivel hacen a lo sumo 1 intercambio.  
Solo la raíz puede hacer hasta log₂(n) intercambios.

Suma total: ≈ n/2·0 + n/4·1 + n/8·2 + ... ≈ **2n** intercambios = O(n).

In [ ]:
def sink(arr, k, n):
    """
    Hundir el elemento en posición k dentro de un heap de tamaño n.
    Versión standalone para Heapsort (no necesita la clase MaxHeap).
    Indexación 1-based: arr[1..n]
    """
    while 2 * k <= n:
        j = 2 * k                           # hijo izquierdo
        if j < n and arr[j] < arr[j + 1]:   # elegir el hijo mayor
            j += 1
        if arr[k] >= arr[j]:                # ya en su lugar
            break
        arr[k], arr[j] = arr[j], arr[k]    # swap con el hijo mayor
        k = j


def heapify(arr, n):
    """
    Construir max-heap in-place aplicando sink de derecha a izquierda.
    Solo se procesan los nodos internos (índices n//2 hasta 1).
    """
    for k in range(n // 2, 0, -1):
        sink(arr, k, n)


# Demostración paso a paso
arr = [None, 4, 1, 3, 2, 9, 7, 8, 5, 6]  # índice 0 sin usar
n = len(arr) - 1

print(f"Arreglo inicial:     {arr[1:]}")
print(f"Nodos internos: posiciones 1 a {n // 2} (posiciones {n//2+1} a {n} son hojas)")
print()

for k in range(n // 2, 0, -1):
    antes = arr[1:].copy()
    sink(arr, k, n)
    print(f"  sink({k}): {antes} → {arr[1:]}")

print(f"\nMax-heap construido: {arr[1:]}")
print(f"Máximo en arr[1] = {arr[1]}  ✓")

In [ ]:
# Comparar costo de heapify con sink vs inserción secuencial con swim
def heapify_swim(arr, n):
    """Construir heap insertando uno a uno (costoso)."""
    comparaciones = 0
    def swim_count(arr, k):
        nonlocal comparaciones
        while k > 1:
            comparaciones += 1
            if arr[k // 2] < arr[k]:
                arr[k // 2], arr[k] = arr[k], arr[k // 2]
                k = k // 2
            else:
                break
    for i in range(2, n + 1):
        swim_count(arr, i)
    return comparaciones

def heapify_sink_count(arr, n):
    """Construir heap con sink desde n//2 (eficiente)."""
    comparaciones = 0
    def sink_count(arr, k, n):
        nonlocal comparaciones
        while 2 * k <= n:
            j = 2 * k
            if j < n:
                comparaciones += 1
                if arr[j] < arr[j + 1]: j += 1
            comparaciones += 1
            if arr[k] >= arr[j]: break
            arr[k], arr[j] = arr[j], arr[k]
            k = j
    for k in range(n // 2, 0, -1):
        sink_count(arr, k, n)
    return comparaciones

ns = [100, 500, 1000, 5000, 10000, 50000]
comp_swim_list = []
comp_sink_list = []

for n in ns:
    datos = random.sample(range(n * 10), n)
    arr1 = [None] + datos[:]
    arr2 = [None] + datos[:]
    comp_swim_list.append(heapify_swim(arr1, n))
    comp_sink_list.append(heapify_sink_count(arr2, n))

plt.figure(figsize=(10, 5))
plt.plot(ns, comp_swim_list, 'ro-', label='heapify con swim — O(n log n)', linewidth=2)
plt.plot(ns, comp_sink_list, 'bo-', label='heapify con sink — O(n)', linewidth=2)
plt.plot(ns, ns, 'g--', label='n (referencia lineal)', linewidth=1.5)
plt.xlabel('n')
plt.ylabel('Comparaciones')
plt.title('Heapify: swim vs sink')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{'n':>8} {'Swim (n log n)':>16} {'Sink (n)':>12} {'Ratio':>8}")
print("-" * 48)
for i, n in enumerate(ns):
    ratio = comp_swim_list[i] / comp_sink_list[i]
    print(f"{n:>8,} {comp_swim_list[i]:>16,} {comp_sink_list[i]:>12,} {ratio:>8.1f}x")

# Sección 3: Fase 2 — Sortdown (12 minutos)

## La idea de sortdown

Una vez que el arreglo es un max-heap, extraemos el máximo **n veces**.

**Truco in-place:** En vez de guardar el máximo en un arreglo separado, lo ponemos en la última posición libre del arreglo original:

```
Estado inicial (heap válido, n=9):
[_, 9, 6, 8, 5, 2, 7, 3, 4, 1]
    ↑máx                    ↑última

Paso 1: swap(1, 9), n=8, sink(1)
[_, 8, 6, 7, 5, 2, 1, 3, 4, |9|]
                             └─ ordenado

Paso 2: swap(1, 8), n=7, sink(1)
[_, 7, 6, 3, 5, 2, 1, 4, |8, 9|]
                          └─ ordenado
...

Paso n: el arreglo completo está ordenado ascendentemente
[_, 1, 2, 3, 4, 5, 6, 7, 8, 9]
```

> 📌 Clave: el máximo extraído **ocupa la posición que acaba de quedar libre** al final del heap. El heap se encoge y el arreglo ordenado crece desde el final.

In [ ]:
def heapsort(arr):
    """
    Heapsort in-place.
    Trabaja sobre una copia interna con indexación 1-based.
    
    Fase 1: heapify con sink — O(n)
    Fase 2: sortdown         — O(n log n)
    Total:                     O(n log n)
    Espacio extra:             O(1)
    """
    n = len(arr)
    # Convertir a 1-based internamente (arreglo de trabajo)
    h = [None] + arr[:]  # h[0] sin usar

    # ── Fase 1: heapify ──────────────────────────────
    for k in range(n // 2, 0, -1):
        sink(h, k, n)

    # ── Fase 2: sortdown ─────────────────────────────
    while n > 1:
        h[1], h[n] = h[n], h[1]   # mover máximo al final
        n -= 1                     # encoger el heap
        sink(h, 1, n)              # restaurar heap-order

    # Copiar de vuelta (0-based)
    for i in range(len(arr)):
        arr[i] = h[i + 1]


# Demostración con tracing
def heapsort_verbose(arr_original):
    arr = arr_original[:]
    n_orig = len(arr)
    h = [None] + arr
    n = n_orig

    print(f"Arreglo inicial:    {h[1:]}")
    print()
    print("── Fase 1: Heapify ──")
    for k in range(n // 2, 0, -1):
        antes = h[1:].copy()
        sink(h, k, n)
        if antes != h[1:]:
            print(f"  sink({k}): {antes} → {h[1:]}")
    print(f"  Max-heap: {h[1:]}")
    print()
    print("── Fase 2: Sortdown ──")
    paso = 1
    while n > 1:
        h[1], h[n] = h[n], h[1]
        n -= 1
        sink(h, 1, n)
        heap_part    = h[1:n+1]
        sorted_part  = h[n+1:]
        print(f"  Paso {paso:2d}: heap={heap_part}  ordenado={sorted_part}")
        paso += 1

    print(f"\nArreglo ordenado:   {h[1:]}")


heapsort_verbose([4, 1, 3, 2, 9, 7, 8, 5, 6])

# Sección 4: Análisis de Complejidad (8 minutos)

## Complejidad de Heapsort

| Fase | Operaciones | Complejidad |
|------|------------|-------------|
| Heapify (fase 1) | n/2 llamadas a sink | **O(n)** |
| Sortdown (fase 2) | n extracciones × sink | **O(n log n)** |
| **Total** | | **O(n log n)** |

## La propiedad única de Heapsort

Heapsort es el **único algoritmo de comparación** que cumple las tres condiciones simultáneamente:

| Propiedad | Heapsort | Mergesort | Quicksort |
|-----------|:--------:|:---------:|:---------:|
| O(n log n) peor caso | ✅ | ✅ | ❌ (O(n²)) |
| O(1) espacio extra | ✅ | ❌ (O(n)) | ✅ |
| Estable | ❌ | ✅ | ❌ |

**Introsort** (usado en C++ `std::sort`) combina Quicksort + Heapsort: usa Quicksort normalmente pero cambia a Heapsort si detecta recursión demasiado profunda (señal del peor caso de Quicksort).

## ¿Por qué Heapsort no se usa más en la práctica?

A pesar de sus garantías teóricas, Heapsort suele ser más lento que Quicksort en la práctica por:

1. **Cache-unfriendly:** Los accesos al heap saltan por posiciones `k`, `2k`, `2k+1` — muy poco localidad de caché.
2. **Muchas comparaciones:** sink compara dos hijos en cada nivel (2 comparaciones por nivel vs 1 en Quicksort).
3. **No aprovecha patrones:** No tiene ventaja en datos parcialmente ordenados (Timsort sí).

> 🎙️ **[PAUSA PROFESOR]** *"Heapsort tiene la mejor garantía teórica: O(n log n) siempre + O(1) espacio. ¿Por qué Python no lo usa?"*  
> *Respuestas esperadas: cache locality, estabilidad, Timsort aprovecha runs.*

In [ ]:
# Verificar que Heapsort funciona correctamente
casos_prueba = [
    ([5, 3, 8, 1, 7, 2, 9, 4, 6], "Aleatorio"),
    ([1, 2, 3, 4, 5, 6, 7, 8, 9], "Ya ordenado"),
    ([9, 8, 7, 6, 5, 4, 3, 2, 1], "Inversamente ordenado"),
    ([3, 1, 4, 1, 5, 9, 2, 6, 5, 3], "Con repetidos"),
    ([42], "Un elemento"),
    ([], "Vacío"),
]

print("Verificación de Heapsort:")
print("-" * 55)
for datos, nombre in casos_prueba:
    arr = datos[:]
    heapsort(arr)
    correcto = arr == sorted(datos)
    estado = "✅" if correcto else "❌"
    print(f"  {estado} {nombre:<28} → {arr}")

# Benchmark Heapsort vs Quicksort vs sorted() para n grande
import sys
sys.setrecursionlimit(50000)

def quicksort_bench(arr, lo=0, hi=None):
    if hi is None: hi = len(arr) - 1
    if lo < hi:
        pivot = arr[hi]
        i = lo - 1
        for j in range(lo, hi):
            if arr[j] <= pivot:
                i += 1
                arr[i], arr[j] = arr[j], arr[i]
        arr[i+1], arr[hi] = arr[hi], arr[i+1]
        p = i + 1
        quicksort_bench(arr, lo, p - 1)
        quicksort_bench(arr, p + 1, hi)

def mergesort_bench(arr):
    if len(arr) <= 1: return arr
    mid = len(arr) // 2
    izq = mergesort_bench(arr[:mid])
    der = mergesort_bench(arr[mid:])
    resultado, i, j = [], 0, 0
    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]: resultado.append(izq[i]); i += 1
        else:                 resultado.append(der[j]); j += 1
    return resultado + izq[i:] + der[j:]

print("\nBenchmark de tiempo (n=5000, datos aleatorios):")
print("-" * 50)

n_bench = 5000
datos_base = random.sample(range(n_bench * 10), n_bench)

resultados_tiempo = {}
for nombre, fn, necesita_copia in [
    ('Heapsort',  lambda a: heapsort(a),       True),
    ('Quicksort', lambda a: quicksort_bench(a), True),
    ('Mergesort', lambda a: mergesort_bench(a), False),
    ('sorted()',  lambda a: sorted(a),          False),
]:
    tiempos = []
    for _ in range(5):
        arr = datos_base[:]
        t0 = time.perf_counter()
        fn(arr)
        tiempos.append(time.perf_counter() - t0)
    promedio = sum(tiempos) / len(tiempos) * 1000
    resultados_tiempo[nombre] = promedio
    print(f"  {nombre:<12} {promedio:>8.2f} ms")

print()
base = resultados_tiempo['sorted()']
for nombre, ms in resultados_tiempo.items():
    print(f"  {nombre:<12} {ms/base:>5.1f}× más lento que sorted()")

# Sección 5: El Gran Cuadro — Comparando los 3 Algoritmos O(n log n) (7 minutos)

In [ ]:
# Visualización comparativa: comportamiento en distintos tipos de input
tipos_input = {
    'Aleatorio':       lambda n: random.sample(range(n*10), n),
    'Ya ordenado':     lambda n: list(range(n)),
    'Casi ordenado':   lambda n: list(range(n-5)) + random.sample(range(n-5, n*2), 5),
    'Inversamente':    lambda n: list(range(n, 0, -1)),
}

ns = [500, 1000, 2000, 3000, 5000]

def medir_tiempo(fn, datos):
    arr = datos[:]
    t0 = time.perf_counter()
    fn(arr)
    return (time.perf_counter() - t0) * 1000

algoritmos = {
    'Heapsort':  lambda a: heapsort(a),
    'Quicksort': lambda a: quicksort_bench(a),
    'sorted()':  lambda a: sorted(a),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colores = {'Heapsort': '#e74c3c', 'Quicksort': '#3498db', 'sorted()': '#2ecc71'}

for idx, (tipo, gen) in enumerate(tipos_input.items()):
    ax = axes[idx // 2][idx % 2]
    for nombre, fn in algoritmos.items():
        tiempos = []
        for n in ns:
            datos = gen(n)
            t = medir_tiempo(fn, datos)
            tiempos.append(t)
        ax.plot(ns, tiempos, 'o-', color=colores[nombre], label=nombre, linewidth=2)
    ax.set_title(f'Input: {tipo}', fontsize=11, fontweight='bold')
    ax.set_xlabel('n')
    ax.set_ylabel('Tiempo (ms)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparación de algoritmos O(n log n) por tipo de input', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Sección 6: Resumen de la Unidad 3 — Ordenamiento (5 minutos)

## Los 6 algoritmos que estudiamos

| Algoritmo | Mejor caso | Caso promedio | Peor caso | Espacio | Estable |
|-----------|:----------:|:------------:|:---------:|:-------:|:-------:|
| Selection Sort | O(n²) | O(n²) | O(n²) | O(1) | ❌ |
| Insertion Sort | O(n) | O(n²) | O(n²) | O(1) | ✅ |
| Shell Sort | O(n log n) | O(n^1.3) | O(n²) | O(1) | ❌ |
| Merge Sort | O(n log n) | O(n log n) | O(n log n) | O(n) | ✅ |
| Quicksort | O(n log n) | O(n log n) | O(n²) | O(log n) | ❌ |
| **Heapsort** | **O(n log n)** | **O(n log n)** | **O(n log n)** | **O(1)** | **❌** |

## ¿Cuándo usar cada uno?

- **¿Necesitas estabilidad?** → Merge Sort o Timsort (`sorted()` de Python)
- **¿RAM limitada y datos aleatorios?** → Quicksort con mediana de 3
- **¿Garantía O(n log n) en peor caso + O(1) espacio?** → Heapsort (o Introsort)
- **¿Datos casi ordenados?** → Insertion Sort o Timsort
- **¿N pequeño (< 50)?** → Insertion Sort directamente

## Conexión con la Unidad 4

En la Unidad 4 (Diccionarios / Búsqueda) veremos estructuras que van más allá del ordenamiento:

- **BST** (Binary Search Tree): O(log n) en búsqueda, inserción y eliminación
- **Red-Black BST**: BST autobalanceado, O(log n) garantizado
- **Hash Tables**: O(1) amortizado, pero sin ordenamiento

La Priority Queue (heap) reaparecerá en gráfos: **Dijkstra** y **Prim** la usan como estructura central.

---

## Referencias

- Sedgewick & Wayne. *Algorithms*, 4ª ed., Sección 2.4
- Williams, J.W.J. (1964). "Algorithm 232: Heapsort". *Communications of the ACM*, 7(6), 347–348.
- Floyd, R.W. (1964). "Algorithm 245: Treesort 3". *Communications of the ACM*, 7(12), 701.
- Cormen et al. *Introduction to Algorithms* (CLRS), Cap. 6